In [ ]:

# 1: Importazioni e Setup
# Qui avviene il cambiamento più importante: importiamo VGG16 e, soprattutto, il suo preprocessing specifico."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "bcf0cf0c",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Librerie importate. Pronto per VGG16.\n"
     ]
    }
   ],
   "source": [
    "import tensorflow as tf\n",
    "import numpy as np\n",
    "import requests\n",
    "from PIL import Image\n",
    "from io import BytesIO\n",
    "\n",
    "# --- MODIFICA 1: Cambio Modulo ---\n",
    "# Non importiamo più da mobilenet_v2. \n",
    "# Importiamo tutto ciò che serve dal pacchetto vgg16.\n",
    "from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions\n",
    "from tensorflow.keras.preprocessing import image as keras_image\n",
    "\n",
    "print(\"Librerie importate. Pronto per VGG16.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0f607ec5",
   "metadata": {},
   "source": [
    "Cella 2: Configurazione\n",
    "Cambiamo l'immagine per testare la rete su un soggetto diverso."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "1a568bab",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "URL impostato: https://images.pexels.com/photos/337909/pexels-photo-337909.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1\n"
     ]
    }
   ],
   "source": [
    "# --- MODIFICA 2: Nuovo Soggetto ---\n",
    "# Cambiamo URL. Proviamo con un'auto sportiva classica per vedere se la riconosce.\n",
    "# Un URL alternativo che solitamente è più permissivo\n",
    "IMMAGINE_URL = \"https://images.pexels.com/photos/337909/pexels-photo-337909.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1\"\n",
    "print(f\"URL impostato: {IMMAGINE_URL}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6403e3ec",
   "metadata": {},
   "source": [
    "Cella 3: Definizione delle Funzioni\n",
    "Qui adattiamo le funzioni per utilizzare la classe VGG16 e richiedere più risultati (top=5)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "8c92efdd",
   "metadata": {},
   "outputs": [],
   "source": [
    "def carica_modello_vgg():\n",
    "    \"\"\"\n",
    "    Carica la rete VGG16 con pesi ImageNet.\n",
    "    Nota: VGG16 è molto più pesante (~500MB) di MobileNet (~14MB).\n",
    "    \"\"\"\n",
    "    print(\"[1/4] Caricamento del modello VGG16 (pesi ImageNet)...\")\n",
    "    print(\"      (Attendi, il download potrebbe richiedere tempo la prima volta)...\")\n",
    "    \n",
    "    # --- MODIFICA 3: Istanziazione VGG16 ---\n",
    "    model = VGG16(weights='imagenet', include_top=True)\n",
    "    return model\n",
    "\n",
    "def ottieni_e_processa_immagine(url):\n",
    "    print(f\"[2/4] Download e processing da: {url}...\")\n",
    "    \n",
    "    response = requests.get(url)\n",
    "    img = Image.open(BytesIO(response.content))\n",
    "    \n",
    "    # VGG16 usa anch'essa 224x224 come standard\n",
    "    img = img.resize((224, 224)) \n",
    "    x = keras_image.img_to_array(img)\n",
    "    x = np.expand_dims(x, axis=0)\n",
    "    \n",
    "    # --- MODIFICA 4: Preprocessing Corretto ---\n",
    "    # Qui stiamo usando la funzione 'preprocess_input' importata da VGG16.\n",
    "    # A differenza di MobileNet (che scala 0-1 o -1,1), VGG16 converte le immagini \n",
    "    # da RGB a BGR e sottrae la media del dataset ImageNet da ogni canale.\n",
    "    # Senza questo passaggio specifico, la predizione sarebbe errata.\n",
    "    x = preprocess_input(x)\n",
    "    \n",
    "    return x, img\n",
    "\n",
    "def classifica_immagine(model, processed_image):\n",
    "    print(\"[3/4] Classificazione in corso...\")\n",
    "    predictions = model.predict(processed_image)\n",
    "    \n",
    "    # --- MODIFICA 5: Top-5 Results ---\n",
    "    # Chiediamo le prime 5 classi invece di 3 per avere più dettagli.\n",
    "    results = decode_predictions(predictions, top=5)[0]\n",
    "    \n",
    "    return results"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0dae1322",
   "metadata": {},
   "source": [
    "Cella 4: Esecuzione (Main)\n",
    "Assembliamo i pezzi ed eseguiamo la classificazione."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "id": "c1c467b6",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[1/4] Caricamento del modello VGG16 (pesi ImageNet)...\n",
      "      (Attendi, il download potrebbe richiedere tempo la prima volta)...\n",
      "[2/4] Download e processing da: https://images.pexels.com/photos/337909/pexels-photo-337909.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1...\n"
     ]
    },
    {
     "data": {